# VGGT With IMU-Initialized Cameras On ADVIO

This notebook compares two runs on the same ADVIO sequence with at most 15 images:

1. `Baseline VGGT`
2. `IMU-initialized camera VGGT`

The key difference is that the second run does **not** just use IMU to choose frames.
It uses IMU orientation priors at the camera stage by fusing the IMU camera initialization with VGGT camera predictions before depth unprojection and point-cloud construction.

Important limitation:
ADVIO gives raw accelerometer and gyroscope, but not clean translation from IMU alone. So this notebook uses IMU to initialize **camera rotation**, while keeping **translation from VGGT**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/3d-reconstruction'
%cd $REPO_DIR

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q plotly huggingface_hub einops

In [ ]:
import os, sys

VGGT_REPO = '/content/vggt'
CKPT_PATH = '/content/model_tracker_fixed_e20.pt'

if not os.path.exists(VGGT_REPO):
    !git clone https://github.com/facebookresearch/vggt.git $VGGT_REPO
if VGGT_REPO not in sys.path:
    sys.path.insert(0, VGGT_REPO)
if not os.path.exists(CKPT_PATH):
    !wget -q --show-progress https://huggingface.co/facebook/VGGT_tracker_fixed/resolve/main/model_tracker_fixed_e20.pt -O $CKPT_PATH

In [ ]:
import os, math

ADVIO_SEQ = '15'
ADVIO_ROOT = f'data/advio-{ADVIO_SEQ}'
ADVIO_ZIP = f'data/advio-{ADVIO_SEQ}.zip'
ADVIO_URL = f'https://zenodo.org/record/1476931/files/advio-{ADVIO_SEQ}.zip'

os.makedirs('data', exist_ok=True)
if not os.path.exists(ADVIO_ZIP):
    !wget -O $ADVIO_ZIP $ADVIO_URL
if not os.path.exists(ADVIO_ROOT):
    !unzip -q -o $ADVIO_ZIP -d data

In [ ]:
from src.datasets.advio import load_advio_iphone_sequence

MAX_IMAGES = 15
PREP_ROOT = f'data/prepared_imu_init/advio-{ADVIO_SEQ}'

seq = load_advio_iphone_sequence(ADVIO_ROOT)
every_nth = max(1, math.ceil(len(seq.frame_timestamps) / MAX_IMAGES))
print('Total frames:', len(seq.frame_timestamps), 'every_nth:', every_nth)

!python prepare_advio_vggt_dataset.py \
  --advio-root $ADVIO_ROOT \
  --output-root $PREP_ROOT \
  --every-nth-frame $every_nth \
  --max-frames $MAX_IMAGES \
  --rotation-thresh-deg 8.0 \
  --translation-thresh-m 0.0 \
  --min-gap 1

!cat $PREP_ROOT/summary.json

In [ ]:
import torch
from vggt.models.vggt import VGGT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16

model = VGGT()
state_dict = torch.load(CKPT_PATH, map_location='cpu')
if 'model' in state_dict:
    state_dict = state_dict['model']
model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()
print('Loaded VGGT on', device, 'dtype', DTYPE)

In [ ]:
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torchvision.transforms as T
from PIL import Image

from src.datasets.advio import load_advio_iphone_sequence
from src.imu.priors import build_frame_pose_priors, build_imu_pose_sequence
from src.imu.fusion import fuse_camera_extrinsics_with_imu

IMG_SIZE = 518
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_images_from_dir(img_dir):
    exts = {'.jpg', '.jpeg', '.png', '.JPG', '.PNG'}
    paths = sorted([p for p in Path(img_dir).iterdir() if p.suffix in exts])
    tensors = []
    for p in paths:
        img = Image.open(p).convert('RGB')
        tensors.append(transform(img))
    return torch.stack(tensors), paths

def load_frame_rows(csv_path):
    rows = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            parts = [x.strip() for x in line.split(',')]
            rows.append((parts[0], float(parts[1])))
    return rows

def build_advio_imu_camera_priors(advio_root, frame_rows):
    seq = load_advio_iphone_sequence(advio_root)
    imu_rows = []
    for acc_row in seq.accelerometer:
        t = float(acc_row[0])
        nearest = np.argmin(np.abs(seq.gyroscope[:, 0] - t))
        imu_rows.append({'timestamp': t, 'gyro': np.asarray(seq.gyroscope[nearest, 1:4], dtype=np.float64)})
    imu_timestamps, imu_poses = build_imu_pose_sequence(imu_rows)
    names, priors = build_frame_pose_priors(frame_rows, imu_timestamps, imu_poses)
    return names, priors

def run_vggt(img_dir, pose_out_dir=None):
    images_tensor, img_paths = load_images_from_dir(img_dir)
    images_input = images_tensor.unsqueeze(0).to(device, dtype=DTYPE)
    t0 = time.time()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=DTYPE):
            aggregated_tokens_list, ps_idx = model.aggregator(images_input)
            from vggt.utils.pose_enc import pose_encoding_to_extri_intri
            pose_enc = model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_input.shape[-2:])
            depth_map, depth_conf = model.depth_head(aggregated_tokens_list, images_input, ps_idx)
            from vggt.utils.geometry import unproject_depth_map_to_point_map
            point_map_unproj = unproject_depth_map_to_point_map(depth_map.squeeze(0), extrinsic.squeeze(0), intrinsic.squeeze(0))
    elapsed = time.time() - t0
    result = {
        'img_paths': img_paths,
        'extrinsic': extrinsic.squeeze(0).float().cpu().numpy(),
        'intrinsic': intrinsic.squeeze(0).float().cpu().numpy(),
        'depth': depth_map.squeeze(0).float().cpu().numpy(),
        'depth_conf': depth_conf.squeeze(0).float().cpu().numpy(),
        'point_map': point_map_unproj.float().cpu().numpy(),
        'elapsed_sec': elapsed,
    }
    if pose_out_dir is not None:
        out = Path(pose_out_dir)
        out.mkdir(parents=True, exist_ok=True)
        for i, path in enumerate(img_paths):
            np.savetxt(out / f'{path.stem}_extrinsic.txt', result['extrinsic'][i], fmt='%.8f')
            np.savetxt(out / f'{path.stem}_intrinsic.txt', result['intrinsic'][i], fmt='%.8f')
    return result

def run_vggt_with_imu_initialized_cameras(img_dir, advio_root, frame_csv, pose_out_dir=None, vision_weight=0.35):
    images_tensor, img_paths = load_images_from_dir(img_dir)
    frame_rows = load_frame_rows(frame_csv)
    frame_rows = frame_rows[:len(img_paths)]
    _, imu_c2w_priors = build_advio_imu_camera_priors(advio_root, frame_rows)

    images_input = images_tensor.unsqueeze(0).to(device, dtype=DTYPE)
    t0 = time.time()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=DTYPE):
            aggregated_tokens_list, ps_idx = model.aggregator(images_input)
            from vggt.utils.pose_enc import pose_encoding_to_extri_intri
            pose_enc = model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_input.shape[-2:])
            depth_map, depth_conf = model.depth_head(aggregated_tokens_list, images_input, ps_idx)

    extrinsic_np = extrinsic.squeeze(0).float().cpu().numpy()
    intrinsic_np = intrinsic.squeeze(0).float().cpu().numpy()
    fused_extrinsic = fuse_camera_extrinsics_with_imu(extrinsic_np, imu_c2w_priors, vision_weight=vision_weight, keep_translation='vision')

    from vggt.utils.geometry import unproject_depth_map_to_point_map
    fused_extrinsic_t = torch.from_numpy(fused_extrinsic).to(device=device, dtype=DTYPE)
    point_map_unproj = unproject_depth_map_to_point_map(depth_map.squeeze(0), fused_extrinsic_t, intrinsic.squeeze(0))

    elapsed = time.time() - t0
    result = {
        'img_paths': img_paths,
        'extrinsic': fused_extrinsic,
        'intrinsic': intrinsic_np,
        'depth': depth_map.squeeze(0).float().cpu().numpy(),
        'depth_conf': depth_conf.squeeze(0).float().cpu().numpy(),
        'point_map': point_map_unproj.float().cpu().numpy(),
        'elapsed_sec': elapsed,
        'imu_c2w_priors': imu_c2w_priors,
        'vision_weight': vision_weight,
    }
    if pose_out_dir is not None:
        out = Path(pose_out_dir)
        out.mkdir(parents=True, exist_ok=True)
        for i, path in enumerate(img_paths):
            np.savetxt(out / f'{path.stem}_extrinsic.txt', result['extrinsic'][i], fmt='%.8f')
            np.savetxt(out / f'{path.stem}_intrinsic.txt', result['intrinsic'][i], fmt='%.8f')
    return result

def merged_point_cloud(result, conf_threshold=1.5, max_points=60000):
    points = result['point_map'].reshape(-1, 3)
    conf = result['depth_conf'].reshape(-1)
    mask = np.isfinite(points).all(axis=1) & (conf > conf_threshold)
    points = points[mask]
    if len(points) > max_points:
        idx = np.random.choice(len(points), size=max_points, replace=False)
        points = points[idx]
    return points

def show_point_cloud(points, title):
    fig = go.Figure(data=[go.Scatter3d(x=points[:,0], y=points[:,1], z=points[:,2], mode='markers', marker=dict(size=1.5, opacity=0.7))])
    fig.update_layout(title=title, scene=dict(aspectmode='data'), height=700, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

In [ ]:
BASELINE_IMG_DIR = f'{PREP_ROOT}/baseline/images'
BASELINE_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/imu_initialized/baseline/poses'
baseline_result = run_vggt(BASELINE_IMG_DIR, BASELINE_POSE_DIR)
baseline_points = merged_point_cloud(baseline_result)
print('Baseline images:', len(baseline_result['img_paths']))
print('Baseline time (s):', baseline_result['elapsed_sec'])
show_point_cloud(baseline_points, 'Baseline VGGT Point Cloud')

In [ ]:
IMU_IMG_DIR = f'{PREP_ROOT}/baseline/images'
IMU_FRAME_CSV = f'{PREP_ROOT}/baseline/metadata/frame_times.csv'
IMU_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/imu_initialized/imu_camera_init/poses'
VISION_WEIGHT = 0.35

imu_init_result = run_vggt_with_imu_initialized_cameras(
    IMU_IMG_DIR,
    ADVIO_ROOT,
    IMU_FRAME_CSV,
    IMU_POSE_DIR,
    vision_weight=VISION_WEIGHT,
)
imu_init_points = merged_point_cloud(imu_init_result)
print('IMU-camera-init images:', len(imu_init_result['img_paths']))
print('IMU-camera-init time (s):', imu_init_result['elapsed_sec'])
print('Vision weight:', VISION_WEIGHT)
show_point_cloud(imu_init_points, 'IMU-Initialized Camera VGGT Point Cloud')

## Optional: Benchmark Baseline vs IMU-Initialized Camera Run

In [ ]:
REPORT_JSON = f'outputs/analysis/advio-{ADVIO_SEQ}_imu_initialized_camera_benchmark.json'
PLOT_DIR = f'outputs/analysis/advio-{ADVIO_SEQ}_imu_initialized_camera'

!python benchmark_advio_vggt.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --imu-guided-pose-dir $IMU_POSE_DIR \
  --output-json $REPORT_JSON

!python visualize_advio_benchmark.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --imu-guided-pose-dir $IMU_POSE_DIR \
  --output-dir $PLOT_DIR

!cat $REPORT_JSON